# TrackViewer Visualization of Filtered STRs and Functional Annotations

This notebook uses the [trackViewer](https://bioconductor.org/packages/trackViewer) (Bioconductor) R package to visualize the **8 filtered STRs** from `Relatorio_STR_Final_Integral.pdf` together with their functional annotations.

**Inputs**
- Local project data: STR coordinates / residual / scRNA-seq expression
- External tracks downloaded by `7.4.3.1_download_external_tracks.sh` into `external_tracks/`

**Reference genome:** hg38 (GRCh38).

In [34]:
suppressPackageStartupMessages({
  library(trackViewer)
  library(GenomicRanges)
  library(rtracklayer)
  library(TxDb.Hsapiens.UCSC.hg38.knownGene)
  library(readr)
})

cat('trackViewer loaded OK\n')

trackViewer loaded OK


## 2. Define the 8 filtered STRs

The 8 STR loci are loaded from the unified scRNA-seq overlap file generated in step 7.3.1
(`../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv`), taking one row per locus.
Genomic coordinates (`start0` 0-based, `end`) are taken from `STR_variants_UCSC_track.bed`
(shipped with this notebook), falling back to `start0 + nchar(motif) * copy` if the BED is absent.


In [35]:
# 2. Define the 8 filtered STRs from the unified overlap CSV + BED coordinates
variants_file <- '../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv'
bed_file <- 'STR_variants_UCSC_track.bed'

if (file.exists(variants_file)) {
  # one row per STR locus (drop duplicated cell_type rows)
  ov <- read_csv(variants_file, show_col_types = FALSE)
  variants <- ov[!duplicated(ov$STRs_ID), ]

  # STRs_ID encodes chr:pos:motif:copy  (pos = 1-based start)
  id_parts <- strsplit(variants$STRs_ID, ':', fixed = TRUE)
  variants$chr    <- vapply(id_parts, function(x) x[1], character(1))
  variants$start1 <- as.integer(vapply(id_parts, function(x) x[2], character(1)))
  variants$motif  <- vapply(id_parts, function(x) x[3], character(1))
  variants$copy   <- as.integer(vapply(id_parts, function(x) x[4], character(1)))
  variants$start0 <- variants$start1 - 1
  variants$gene   <- variants$gene_name
  variants$allele2 <- variants$allele2_est
  variants$group  <- ifelse(tolower(variants$group) == 'case', 'Case', 'Control')

  # BED coordinates for start0 / end (0-based start, 1-based-style end)
  if (file.exists(bed_file)) {
    bed <- read.delim(bed_file, comment.char = '#', header = FALSE, fill = TRUE,
                      col.names = c('chr', 'start0', 'end', 'name'))
    bed <- bed[grepl('^chr', bed$chr), ]
    bed$STRs_ID <- sub('^[^_]+_', '', bed$name)  # name = <gene>_<STRs_ID>
    bed <- bed[bed$STRs_ID %in% variants$STRs_ID, c('STRs_ID', 'end')]
    variants$end <- bed$end[match(variants$STRs_ID, bed$STRs_ID)]
    if (anyNA(variants$end)) {
      warning('Some loci missing from BED; estimating end from motif length')
      na <- is.na(variants$end)
      variants$end[na] <- variants$start0[na] + nchar(variants$motif[na]) * variants$copy[na]
    }
  } else {
    cat('BED file not found; estimating end from motif length * copy\n')
    variants$end <- variants$start0 + nchar(variants$motif) * variants$copy
  }

  cat('Loaded', nrow(variants), 'STR loci from', variants_file, '\n')
} else {
  stop('Overlap CSV not found: ', variants_file)
}

print(variants[, c('STRs_ID', 'gene', 'abs_res', 'allele2', 'group', 'chr', 'start0', 'end')])

# GRanges of the STRs (1-based start)
gr_strs <- GRanges(seqnames = variants$chr,
                   ranges = IRanges(start = variants$start0 + 1, end = variants$end),
                   strand = '*',
                   STRs_ID = variants$STRs_ID,
                   gene = variants$gene,
                   abs_res = variants$abs_res,
                   allele2 = variants$allele2,
                   group = variants$group,
                   motif = variants$motif)
names(gr_strs) <- variants$gene
gr_strs

Loaded 8 STR loci from ../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv 
# A tibble: 8 × 8
  STRs_ID              gene       abs_res allele2 group   chr      start0    end
  <chr>                <chr>        <dbl>   <dbl> <chr>   <chr>     <dbl>  <int>
1 chr3:76185195:AT:11  ROBO2         26.2    18.5 Control chr3   76185194 7.62e7
2 chr10:60288889:AC:19 ANK3          20.6    16.8 Control chr10  60288888 6.03e7
3 chr5:22138845:AT:16  CDH12         61.1    22.8 Control chr5   22138844 2.21e7
4 chr6:123976248:AT:8  NKAIN2        43.7    21.2 Control chr6  123976247 1.24e8
5 chr15:47551673:GT:14 SEMA6D        33.2    14.4 Case    chr15  47551672 4.76e7
6 chr1:211045041:AT:9  KCNH1         30.1    31.2 Control chr1  211045040 2.11e8
7 chr6:73097142:AT:9   KCNQ5         23.6    14.8 Control chr6   73097141 7.31e7
8 chr1:76143392:GT:16  ST6GALNAC3    13.4    24.9 Control chr1   76143391 7.61e7


GRanges object with 8 ranges and 6 metadata columns:
             seqnames              ranges strand |              STRs_ID
                <Rle>           <IRanges>  <Rle> |          <character>
       ROBO2     chr3   76185195-76185206      * |  chr3:76185195:AT:11
        ANK3    chr10   60288889-60288908      * | chr10:60288889:AC:19
       CDH12     chr5   22138845-22138861      * |  chr5:22138845:AT:16
      NKAIN2     chr6 123976248-123976256      * |  chr6:123976248:AT:8
      SEMA6D    chr15   47551673-47551687      * | chr15:47551673:GT:14
       KCNH1     chr1 211045041-211045050      * |  chr1:211045041:AT:9
       KCNQ5     chr6   73097142-73097151      * |   chr6:73097142:AT:9
  ST6GALNAC3     chr1   76143392-76143408      * |  chr1:76143392:GT:16
                    gene   abs_res   allele2       group       motif
             <character> <numeric> <numeric> <character> <character>
       ROBO2       ROBO2   26.1541     18.47     Control          AT
        ANK3        

## 3. Load external tracks

Files are expected in `external_tracks/` (created by the download script). Each track is a helper function that returns a `trackViewer` feature track (importing bigWig/BED via `rtracklayer::import`).

In [36]:
ext_dir <- 'external_tracks'
stopifnot(dir.exists(ext_dir))

# Window covering all 8 loci (+/- 20 kb) so we import only what is plotted
# instead of the whole-genome bigWig (avoids multi-GB in-memory GRanges).
win_all <- if (exists('gr_strs') && length(gr_strs) > 0) {
  reduce(resize(gr_strs, width = 40000, fix = 'center'))
} else {
  NULL  # fall back to whole-genome import if cell 3 did not produce loci
}

# Track building helpers -----------------------------------------------------
# class is 'track' (lowercase); color lives in trackStyle (setTrackStyleParam)

# quick magic-byte sanity checks to avoid importing error pages saved as .bw
is_bigwig <- function(path) {
  con <- file(path, 'rb'); on.exit(close(con))
  magic <- readBin(con, 'raw', n = 4)
  length(magic) == 4 && (
    identical(magic, as.raw(c(0x26, 0xfc, 0x8f, 0x88))) ||  # bigWig LE
    identical(magic, as.raw(c(0x88, 0x8f, 0xfc, 0x26)))     # bigWig BE
  )
}

is_text_error <- function(path) {
  con <- file(path, 'rb'); on.exit(close(con))
  magic <- readBin(con, 'raw', n = 2)
  if (length(magic) == 2 && identical(magic, as.raw(c(0x1f, 0x8b)))) return(FALSE)
  head <- readChar(con, nchars = 200, useBytes = TRUE)
  grepl('<html|<!DOCTYPE', head, ignore.case = TRUE, useBytes = TRUE)
}

import_bw <- function(path, name, color, win = NULL) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  if (!is_bigwig(path)) {
    warning('Not a valid bigWig (magic bytes), skipping: ', path)
    return(NULL)
  }
  gr <- rtracklayer::import(path, which = win)
  if (length(gr) == 0) return(NULL)
  if (is.null(gr$score)) mcols(gr)$score <- 0
  tr <- new("track", dat = gr, type = "data", format = "BigWig", name = name)
  setTrackStyleParam(tr, "color", color)
  tr
}

import_bed <- function(path, name, color, win = NULL) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  if (is_text_error(path)) {
    warning('Looks like an error/HTML page, skipping: ', path)
    return(NULL)
  }
  gr <- rtracklayer::import(path, format = 'BED', which = win)
  if (length(gr) == 0) return(NULL)
  if (is.null(gr$score)) mcols(gr)$score <- 1
  tr <- new("track", dat = gr, type = "data", format = "BED", name = name)
  setTrackStyleParam(tr, "color", color)
  tr
}

# Individual annotation files -----------------------------------------------
tracks_external <- list(
  CTCF   = import_bw(file.path(ext_dir, 'CTCF_ENCFF910VLV.bw'),       'CTCF ChIP-seq',    '#D7301F', win_all),
  DNase  = import_bw(file.path(ext_dir, 'DNase_brain.bw'),            'DNase',            '#E08214', win_all),
  H3K27ac= import_bw(file.path(ext_dir, 'H3K27ac_brain.bw'),          'H3K27ac',          '#8073AC', win_all),
  RemapD = import_bw(file.path(ext_dir, 'remap2022_density_hg38.bw'), 'ReMap Density',    '#4575B4', win_all)
)

# ReMap TF peak tracks (any available) - keyed with a _peaks suffix so the
# CTCF ReMap peaks do not overwrite the CTCF ChIP-seq signal track.
tf_names <- c('CTCF','REST','KLF9','GATA2','ZNF384','MNT')
tf_files <- c(
  'remap2022_ctcf_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_rest_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_klf9_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_gata2_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_znf384_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_mnt_all_macs2_hg38_v1_0.bed.gz'
)
for (i in seq_along(tf_names)) {
  tr <- import_bed(file.path(ext_dir, tf_files[i]), tf_names[i], '#238B45', win_all)
  if (!is.null(tr)) tracks_external[[paste0(tf_names[i], '_peaks')]] <- tr
}

cat('External tracks loaded:', sum(!vapply(tracks_external, is.null, logical(1))), '/', length(tracks_external), '
')

External tracks loaded: 10 / 10 


## 4. Load local project data (scRNA-seq expression)

Optional overlay: LogFC per cell type from the unified overlap CSV (`../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv`), shipped in this repo, used to color/adjust the STR features.


In [37]:
scrna_file <- '../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv'
if (file.exists(scrna_file)) {
  scrna <- read_csv(scrna_file, show_col_types = FALSE)
  scrna <- scrna[!is.na(scrna$LogFC) & !is.na(scrna$STRs_ID), ]
  cat('scRNA overlap rows:', nrow(scrna), '\n')
  print(unique(scrna[, c('gene_name','STRs_ID','source_tissue','LogFC')]))
} else {
  cat('scRNA overlap file not found; skipping overlay\n')
  scrna <- NULL
}

scRNA overlap rows: 15 
# A tibble: 15 × 4
   gene_name  STRs_ID              source_tissue LogFC
   <chr>      <chr>                <chr>         <dbl>
 1 ROBO2      chr3:76185195:AT:11  lung          -93.1
 2 ANK3       chr10:60288889:AC:19 lung          -45.0
 3 ANK3       chr10:60288889:AC:19 lung          -21.7
 4 ANK3       chr10:60288889:AC:19 lung          -19.9
 5 ANK3       chr10:60288889:AC:19 lung          -22.9
 6 ANK3       chr10:60288889:AC:19 lung          -35.3
 7 CDH12      chr5:22138845:AT:16  brain          36.5
 8 NKAIN2     chr6:123976248:AT:8  brain          77.8
 9 NKAIN2     chr6:123976248:AT:8  brain         147. 
10 SEMA6D     chr15:47551673:GT:14 brain         -33.4
11 KCNH1      chr1:211045041:AT:9  brain         -47.9
12 ROBO2      chr3:76185195:AT:11  brain          69.0
13 ROBO2      chr3:76185195:AT:11  brain          65.9
14 KCNQ5      chr6:73097142:AT:9   brain         -55.4
15 ST6GALNAC3 chr1:76143392:GT:16  brain          35.0


## 5. Gene model track

Build a gene track per locus from the TxDb (hg38 knownGene).

In [38]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

# geneModelFromTxdb returns a LIST of transcript tracks (one per transcript)
gene_track_for <- function(gr_variant) {
  # extend window a bit to capture the gene context
  win <- gr_variant
  start(win) <- start(win) - 20000
  end(win)   <- end(win)   + 20000
  
  gt <- geneModelFromTxdb(txdb, gr = win)
  
  # NOVO: Se houver múltiplos transcritos empilhados, mantemos apenas o primeiro 
  # para garantir um layout minimalista e focado no locus.
  if (length(gt) > 1) {
    gt <- gt[1] 
  }
  
  return(gt)
}

cat('Gene model helper ready\n')

Gene model helper ready


vers. gemini

## 6. Variant track with annotations

Each STR is drawn as a feature with height scaled by absolute residual and colored by group.

vers. gemini

In [46]:
make_str_track <- function(gr_variant) {
  # Janela centralizada de 500pb para renderizar o bloco do STR
  center_pos <- start(gr_variant) + floor(width(gr_variant) / 2)
  pos_seq <- seq(center_pos - 250, center_pos + 250, by = 10)
  
  gr <- GRanges(
    seqnames = seqnames(gr_variant),
    ranges = IRanges(start = pos_seq, width = 10),
    strand = "*"
  )
  mcols(gr)$score <- 1
  
  track_label <- paste0("STR: ", gr_variant$gene)
  
  # Usamos type = "data" (único tipo aceito para faixas BED customizadas)
  tr <- new("track", dat = gr, type = "data", format = "BED", name = track_label)
  
  # Cor baseada no grupo (Case = Vermelho, Control = Azul)
  col_group <- ifelse(gr_variant$group == 'Case', '#C62828', '#1565C0')
  setTrackStyleParam(tr, "color", col_group)
  setTrackStyleParam(tr, "height", 0.08)
  
  return(tr)
}

make_scrna_track <- function(scrna, gr_variant) {
  if (is.null(scrna)) return(NULL)
  sub <- scrna[scrna$STRs_ID == gr_variant$STRs_ID, ]
  if (nrow(sub) == 0) return(NULL)
  
  mean_logfc <- mean(sub$LogFC, na.rm = TRUE)
  if (is.na(mean_logfc) || mean_logfc == 0) return(NULL)
  
  center_pos <- start(gr_variant) + floor(width(gr_variant) / 2)
  pos_seq <- seq(center_pos - 250, center_pos + 250, by = 10)
  
  gr <- GRanges(
    seqnames = seqnames(gr_variant),
    ranges = IRanges(start = pos_seq, width = 10),
    strand = "*"
  )
  mcols(gr)$score <- mean_logfc
  
  tr <- new("track", dat = gr, type = "data", format = "BED",
            name = paste0(gr_variant$gene, ' LogFC'))
  
  setTrackStyleParam(tr, "color", '#6A51A3')
  setTrackStyleParam(tr, "height", 0.08)
  
  return(tr)
}

make_scrna_track <- function(scrna, gr_variant) {
  if (is.null(scrna)) return(NULL)
  sub <- scrna[scrna$STRs_ID == gr_variant$STRs_ID, ]
  if (nrow(sub) == 0) return(NULL)
  
  # Expandimos visualmente para 300pb para ser renderizado na janela de 40kb
  gr <- resize(gr_variant, width = 300, fix = "center")
  mcols(gr)$score <- mean(sub$LogFC, na.rm = TRUE)
  tr <- new("track", dat = gr, type = "data", format = "BED",
            name = paste0(gr_variant$gene, ' LogFC'))
  setTrackStyleParam(tr, "color", '#6A51A3')
  setTrackStyleParam(tr, "height", 0.08)
  tr
}

In [75]:
dir.create('results', showWarnings = FALSE)

make_str_track <- function(gr_variant, clean_id) {
  center_pos <- start(gr_variant) + floor(width(gr_variant) / 2)
  pos_seq <- seq(center_pos - 100, center_pos + 100, by = 10)
  
  gr <- GRanges(
    seqnames = seqnames(gr_variant),
    ranges = IRanges(start = pos_seq, width = 10),
    strand = "*"
  )
  mcols(gr)$score <- 1
  
  tr <- new("track", dat = gr, type = "data", format = "BED", name = clean_id)
  
  col_group <- ifelse(gr_variant$group == 'Case', '#C62828', '#1565C0')
  setTrackStyleParam(tr, "color", col_group)
  setTrackStyleParam(tr, "height", 0.08)
  setTrackYaxisParam(tr, "draw", FALSE)
  
  return(tr)
}

render_variant <- function(view_window, track_list, out_png) {
  png(out_png, width = 1400, height = 900, res = 150)
  
  viewer_style <- trackViewerStyle()
  viewer_style@margin <- c(0.05, 0.18, 0.05, 0.02)
  
  viewTracks(track_list, 
             gr = view_window,
             viewerStyle = viewer_style,
             autoOptimizeStyle = TRUE)
             
  dev.off()
  cat('saved:', out_png, '\n')
}

track_list_names <- names(tracks_external)

for (i in seq_len(nrow(variants))) {
  gr_v <- gr_strs[i]

  # Regex para transformar o ID da STR
  raw_id <- gr_v$STRs_ID
  clean_id <- gsub("\\.", ":", sub("^[^:]+:[0-9]+:", "", raw_id))

  trackList <- list()
  trackNames <- character(0)

  # Janela de visualização 
  view_window <- resize(gr_v, width = 1000, fix = 'center')

  # 1. Gene model (Fica na base) com o nome do gene substituindo o transcrito
  gt <- tryCatch(gene_track_for(gr_v), error = function(e) NULL)
  if (!is.null(gt)) {
    names(gt) <- gr_v$gene
    trackList <- c(trackList, gt)
    trackNames <- c(trackNames, names(gt))
  }

  # 2. External functional tracks (Ficam no meio)
  for (nm in track_list_names) {
    tr <- tracks_external[[nm]]
    if (!is.null(tr)) {
      tr_sub <- tr
      tr_sub@dat <- subsetByOverlaps(tr@dat, view_window)
      
      has_peaks <- length(tr_sub@dat) > 0 && max(abs(tr_sub@dat$score), na.rm = TRUE) > 1
      
      if (has_peaks) {
        trackList[[length(trackList) + 1]] <- tr_sub
        trackNames <- c(trackNames, nm)
      }
    }
  }

  # 3. STR variant (No final para ficar no topo) passando o clean_id corretamente
  trackList[[length(trackList) + 1]] <- make_str_track(gr_v, clean_id)
  trackNames <- c(trackNames, clean_id)

  names(trackList) <- trackNames

  # Estética limpa para os eixos Y
  for (j in seq_along(trackList)) {
    setTrackStyleParam(trackList[[j]], "ylabgp", list(cex = 0.6)) 
    setTrackYaxisParam(trackList[[j]], "gp", list(cex = 0.5))
  }

  out_png <- sprintf('results/trackviewer_%s.png', gr_v$gene)
  
  render_variant(view_window, trackList, out_png)
}

cat('\nAll variant figures generated under results/\n')

Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
"genome version information is not available for this TxDb object"


saved: results/trackviewer_ROBO2.png 


Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
"genome version information is not available for this TxDb object"


saved: results/trackviewer_ANK3.png 


Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
"genome version information is not available for this TxDb object"


...

saved: results/trackviewer_KCNH1.png 


Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
"genome version information is not available for this TxDb object"


saved: results/trackviewer_KCNQ5.png 


Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
"genome version information is not available for this TxDb object"


saved: results/trackviewer_ST6GALNAC3.png 

All variant figures generated under results/


## 7. Alinhamento de reads (BAM) com Gviz — amostra com variante vs controle sem a variante

Para cada um dos 8 loci de STR, plota o alinhamento (reads) do BAM indexado da amostra que carrega a variante
(coluna `outlier_samples` do CSV, já carregada em `variants`) contra um BAM controle **sem a variante**
(selecionado a partir de `STRs_normalized_residuals.tsv`: outra amostra do mesmo locus, não-outlier, existente como `.bam`+`.bai` no diretorio `recal/`).
A regiao exata do STR (start–end) e sombreada com `HighlightTrack` (Gviz). Janela de zoom estreita (padrao 300 pb) centrada no STR.


In [ ]:
suppressPackageStartupMessages({
  library(Gviz)
  library(GenomicRanges)
  library(Rsamtools)
  library(TxDb.Hsapiens.UCSC.hg38.knownGene)
})
options(ucscChromosomeNames = TRUE)
if (!exists("txdb")) txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
stopifnot(exists("variants"))

# ---- inputs (edite se preciso) ----
bam_dir   <- "/storage/users/tulio/Projeto_Luy_COVID/results/recal/"
norm_file <- "/storage2/matheusbomfim/projects/git_repos/STRs_COVID_Analysis/5_dbscan/norm_test/STRs_normalized_residuals.tsv"
win_half  <- 150  # 300 pb de janela; use 250 p/ 500 pb
set.seed(20260813)

stopifnot(dir.exists(bam_dir), file.exists(norm_file))

norm <- readr::read_tsv(norm_file, show_col_types = FALSE)
norm$STRs_ID  <- trimws(norm$STRs_ID)
norm$sample_id <- trimws(norm$sample_id)
norm$resid <- suppressWarnings(as.numeric(norm$allele2_residuals))

all_bams <- list.files(bam_dir, pattern = "\\.bam$", full.names = TRUE)
all_bams <- all_bams[file.exists(sub("\\.bam$", ".bai", all_bams))]

dir.create('results_reads_alignment_gviz', showWarnings = FALSE)

verify_tab <- data.frame(
  gene = character(), STRs_ID = character(),
  variant_bam = character(), variant_exists = logical(),
  variant_indexed = logical(), variant_reads = integer(),
  control_bam = character(), control_exists = logical(),
  stringsAsFactors = FALSE)

for (i in seq_len(nrow(variants))) {
  v_chr   <- variants$chr[i]
  v_start <- variants$start0[i] + 1   # 1-based
  v_end   <- variants$end[i]
  v_gene  <- variants$gene[i]
  v_bam   <- file.path(bam_dir, variants$outlier_samples[i])

  center  <- v_start + floor((v_end - v_start) / 2)
  win_min <- max(1, center - win_half)
  win_max <- center + win_half
  win_gr  <- GRanges(v_chr, IRanges(win_min, win_max))

  # --- verificacao do BAM da amostra com a variante ---
  v_exists  <- file.exists(v_bam)
  v_indexed <- file.exists(sub("\\.bam$", ".bai", v_bam))
  v_reads   <- 0L
  if (v_exists && v_indexed) {
    v_reads <- countBam(v_bam, param = ScanBamParam(which = win_gr))$records
  }

  # --- controle: outra amostra do MESMO locus, nao-outlier, com BAM indexado ---
  # sem resid_cutoff: ordena candidatos pelo menor |allele2_residuals| (mais normal)
  # e pega o topo; se nao houver com resid valido, cai para qualquer outro BAM.
  ctrl_row <- NULL
  sub <- norm[norm$STRs_ID == variants$STRs_ID[i], ]
  if (nrow(sub) > 0) {
    cand <- sub[!is.na(sub$sample_id) & sub$sample_id != variants$outlier_samples[i], ]
    cand <- cand[!is.na(cand$resid), ]
    if (nrow(cand) > 0) {
      cand_bam <- file.path(bam_dir, cand$sample_id)
      cand$in_dir <- file.exists(cand_bam) &
        file.exists(sub("\\.bam$", ".bai", cand_bam))
      cand <- cand[cand$in_dir, ]
      if (nrow(cand) > 0) {
        cand <- cand[order(abs(cand$resid)), ]
        ctrl_row <- cand[1, ]
      }
    }
  }
  if (is.null(ctrl_row)) {
    others <- setdiff(all_bams, v_bam)
    if (length(others) > 0) ctrl_row <- data.frame(sample_id = basename(sample(others, 1)))
  }

  c_bam    <- if (!is.null(ctrl_row)) file.path(bam_dir, ctrl_row$sample_id) else NA_character_
  c_exists <- !is.na(c_bam) && file.exists(c_bam)

  verify_tab <- rbind(verify_tab, data.frame(
    gene = v_gene, STRs_ID = variants$STRs_ID[i],
    variant_bam = basename(v_bam), variant_exists = v_exists,
    variant_indexed = v_indexed, variant_reads = v_reads,
    control_bam = if (is.na(c_bam)) NA_character_ else basename(c_bam),
    control_exists = c_exists))

  if (!(v_exists && v_indexed)) { warning('BAM da variante ausente/sem indice: ', v_bam); next }
  if (v_reads == 0) warning('Sem reads na janela para o BAM da variante: ', v_bam)
  if (!c_exists) { warning('Sem controle valido para ', v_gene); next }

  # --- tracks Gviz ---
  gAxis <- GenomeAxisTrack(name = 'Coordenadas (hg38)')

  track_case <- AlignmentsTrack(range = v_bam, genome = 'hg38',
      chromosome = v_chr, name = paste0('Variant: ', sub('\\.bam$', '', basename(v_bam))),
      type = c('pileup'), fill.reads = '#C62828', col.sameseq = '#C62828')
  track_control <- AlignmentsTrack(range = c_bam, genome = 'hg38',
      chromosome = v_chr, name = paste0('Control: ', sub('\\.bam$', '', basename(c_bam))),
      type = c('pileup'), fill.reads = '#1565C0', col.sameseq = '#1565C0')

  gene_track <- tryCatch(
    GeneRegionTrack(txdb, chromosome = v_chr, start = win_min, end = win_max,
        name = v_gene, transcriptAnnotation = 'symbol', fill = '#424242'),
    error = function(e) GeneRegionTrack(txdb, chromosome = v_chr,
        start = win_min, end = win_max, name = v_gene,
        transcriptAnnotation = 'transcript', fill = '#424242'))

  ht <- HighlightTrack(trackList = list(track_case, track_control, gene_track),
      start = v_start, end = v_end, chromosome = v_chr,
      fill = '#FFE082', col = '#FFB300')

  out_png <- sprintf('results_reads_alignment_gviz/gviz_%s.png', v_gene)
  png(out_png, width = 1600, height = 1000, res = 150)
  plotTracks(list(gAxis, ht), from = win_min, to = win_max,
      chromosome = v_chr, sizes = c(0.1, 0.9), title.width = 1.3)
  dev.off()
  cat('Plot Gviz gerado:', v_gene, '| variant:', basename(v_bam),
      '| control:', basename(c_bam), '\n')
}

print(verify_tab)
